# 00 — Setup

One-time setup: install deps, authenticate via GitHub CLI, prove the API works.

_Full narration in `speaker_notes/00_setup.md`._

## 1. Install dependencies

In [ ]:
%pip install --quiet openai python-dotenv

## 2. Authentication

The notebooks authenticate via **GitHub CLI**.

### One-time setup
Install [GitHub CLI](https://cli.github.com/), then:

```bash
gh auth login
```

### How it works
1. **Cached token**: if `~/.claude/gh_models_token.json` holds a non-expired token, it's reused.
2. **GitHub CLI**: otherwise `gh auth token` is invoked and the result is cached for 1 hour.

The cache file is restricted to your user (`chmod 0o600`). `gh` handles its own token refresh, so cache misses just re-fetch.

In [1]:
from llm_client import _get_github_models_token, TOKEN_CACHE

try:
    token = _get_github_models_token()
    masked = token[:7] + "..." + token[-4:] if len(token) > 11 else "***"
    print(f"\nAuth OK. Token: {masked}")
    print(f"Cache location: {TOKEN_CACHE}")
    if TOKEN_CACHE.exists():
        import json
        from datetime import datetime
        cache_data = json.loads(TOKEN_CACHE.read_text())
        exp_time = datetime.fromtimestamp(cache_data['expires_at'])
        print(f"Cached until: {exp_time.isoformat()}")
except Exception as e:
    print(f"Auth error: {e}")
    raise


Looking for authentication via 'gh' CLI...
Using token from 'gh' CLI.

Auth OK. Token: gho_aqQ...dVTn
Cache location: C:\Users\yunliu1\.claude\gh_models_token.json
Cached until: 2026-06-11T14:57:37.618228


## 2. Hello, world

Single chat completion through the shared `get_client()`.

In [2]:
from llm_client import get_client, DEFAULT_MODEL

client = get_client()

resp = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Say hello in five words."}],
)
print(resp.choices[0].message.content)

Hello, nice to meet you.


If you got a five-word greeting back, the SDK, endpoint, and token all work.

Common failures: `401` → token invalid; `429` → rate limited; connection refused → corporate proxy.

## 3. Hello, tool

Prove function-calling works. Every later lesson rides on this 3-message dance:
`user → assistant(tool_calls) → tool(result) → assistant(text)`.

In [3]:
import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current temperature for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            },
        },
    }
]

messages = [{"role": "user", "content": "What is the temperature in Tokyo right now?"}]

resp = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages, tools=tools)
msg = resp.choices[0].message
print("finish_reason:", resp.choices[0].finish_reason)
print("tool_calls:", msg.tool_calls)

finish_reason: tool_calls
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_GX2Hvw3zfqwhi3a5Z4BDSqgV', function=Function(arguments='{"city":"Tokyo"}', name='get_weather'), type='function')]


Append the tool result and call again — this time the model finishes with `stop`.

In [4]:
# Append the assistant turn that asked for the tool
messages.append(msg.model_dump(exclude_none=True))

# Append a tool-result message for each tool_call (here, just one)
for call in msg.tool_calls:
    args = json.loads(call.function.arguments)
    fake_result = f"22°C in {args['city']}"
    messages.append({
        "role": "tool",
        "tool_call_id": call.id,
        "content": fake_result,
    })

resp2 = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages, tools=tools)
print(resp2.choices[0].message.content)

The current temperature in Tokyo is 22°C.


## You're ready

Open `01_foundations.ipynb` next.